In [ ]:
"""
Regression with OLS Feature Selection Predicting MAP
Dylan Polvi

Proposed Hypothesis
My hypothesis is that a regression model that uses a smaller set of physiologically justified features
that's selected with OLS p-values will perform better than the original model, with it using all features provided.
This model will run an OLS regression, view the p-value for each predictor, then keep only the features
that are both statistically significant and have a clear relationship with blood pressure regulation.
I think this will perform better than the original model as it removes features that could be weak or noisy.
Additionally, it would allow the model to be easier to interpret, as there are both less features, as well as the remaining features being statistically valid for inclusion.

Acceptance Thresholds
The alternate model will be considered verified if several criteria are met.
The first is having the RMSE be equal to or higher than the original, or it doesn’t increase past 1 mmHG.
This makes sure that the prediction error doesn’t get worse, while also allowing for some amount of variation due to the decreased feature count.
Next, the R^2 should be equal to or higher than the baseline, or within 0.02 if the RMSE improves, which makes sure that the model has a similar proportion of variance.
Next, the number of features should decrease compared to the original, confirming that there is a real difference between the models.
Next, all features that are kept should be statistically significant, and should therefore have a p value below 0.05.
Finally, all retained features should have some physiological justification to ensure it is relevant to the situation.

Evidence Placeholders
There will be a comparison table comparing the baseline model and the new reduced feature model.
Specifically, the number of features used by each model, the R^2 values, the RMSE values, the MAE values will all be displayed to allow for easy comparisons between the models to find the more effective one.
Additionally, there will also be an OLS table comparing p values for each feature, allowing for the viewer to see why each feature was kept or tossed.
Beside this, there will also be a list of the features that remain in the new model and their physiological relevancy.

"""

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from datasetup import load_data
from feature_policy import REGRESSION_TARGET as TARGET, regression_features

RANDOM_SEED = 42
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)

train, val, test = load_data()

features = regression_features(train)

train = train.dropna(subset=[TARGET]).reset_index(drop=True)
val = val.dropna(subset=[TARGET]).reset_index(drop=True)
test = test.dropna(subset=[TARGET]).reset_index(drop=True)

X_train, y_train, groups_train = train[features], train[TARGET], train["patient_id"]
X_val, y_val = val[features], val[TARGET]
X_test, y_test = test[features], test[TARGET]

print(f"Initial features ({len(features)}): {features}")
print(f"Train rows: {len(X_train)} ({groups_train.nunique()} patients)")
print(f"Val rows:   {len(X_val)}")
print(f"Test rows:  {len(X_test)}")

imputer = SimpleImputer(strategy="mean")
scaler = StandardScaler()

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=features,
    index=X_train.index,
)

X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=features,
    index=X_val.index,
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=features,
    index=X_test.index,
)

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=features,
    index=X_train.index,
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_imp),
    columns=features,
    index=X_val.index,
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=features,
    index=X_test.index,
)

def backward_elimination_ols(X, y, p_thresh=0.05):
    selected = list(X.columns)

    while len(selected) > 1:
        X_ols = sm.add_constant(X[selected], has_constant="add")
        model = sm.OLS(y, X_ols).fit()

        pvalues = model.pvalues.drop("const", errors="ignore")
        worst_p = pvalues.max()
        worst_feature = pvalues.idxmax()

        if worst_p > p_thresh:
            selected.remove(worst_feature)
        else:
            break

    final_X = sm.add_constant(X[selected], has_constant="add")
    final_model = sm.OLS(y, final_X).fit()

    return selected, final_model

selected_features, ols_model = backward_elimination_ols(
    X_train_scaled,
    y_train,
    p_thresh=0.05,
)

print(f"\nSelected features ({len(selected_features)}): {selected_features}")
print(ols_model.summary())

X_val_selected = sm.add_constant(X_val_scaled[selected_features], has_constant="add")
X_test_selected = sm.add_constant(X_test_scaled[selected_features], has_constant="add")

val_preds = ols_model.predict(X_val_selected)
test_preds = ols_model.predict(X_test_selected)

print(f"\nValidation")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, val_preds)):.3f} mmHg")
print(f"MAE : {mean_absolute_error(y_val, val_preds):.3f} mmHg")
print(f"R^2 : {r2_score(y_val, val_preds):.4f}")

print(f"\nTest")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, test_preds)):.3f} mmHg")
print(f"MAE : {mean_absolute_error(y_test, test_preds):.3f} mmHg")
print(f"R^2 : {r2_score(y_test, test_preds):.4f}")

metrics = pd.DataFrame([{
    "split": "val",
    "rmse": np.sqrt(mean_squared_error(y_val, val_preds)),
    "mae": mean_absolute_error(y_val, val_preds),
    "r2": r2_score(y_val, val_preds),
    "num_features": len(selected_features),
}, {
    "split": "test",
    "rmse": np.sqrt(mean_squared_error(y_test, test_preds)),
    "mae": mean_absolute_error(y_test, test_preds),
    "r2": r2_score(y_test, test_preds),
    "num_features": len(selected_features),
}])

metrics.to_csv(
    os.path.join(RESULTS_DIR, "ols_feature_selection_regression_metrics.csv"),
    index=False,
)

coef_table = pd.DataFrame({
    "Feature": ols_model.params.index,
    "Coefficient": ols_model.params.values,
    "P_value": ols_model.pvalues.values,
})

coef_table.to_csv(
    os.path.join(RESULTS_DIR, "ols_feature_selection_coefficients.csv"),
    index=False,
)

print(f"\nSaved: {os.path.join(RESULTS_DIR, 'ols_feature_selection_regression_metrics.csv')}")